# Compare `MyIEEE14_initialized` and Dynawo IEEE14 `IEEE14DisconnectLine`

This notebook mirrors `Compare_MyIEEE14_IEEE14NoEvent.ipynb`, but compares the generated initialized IEEE14 model against the Dynawo Modelica library example `Dynawo.Examples.IEEE14.TestCases.IEEE14DisconnectLine`.

The initialized package path is configurable in the first code cell. By default it uses the working copy created by the OpenModelica initialization workflow.


In [ ]:
using OMJulia
using DataFrames
using Printf

# --- Configuration ---

# Local root for the OpenModelica-only notebooks
DEFAULT_ROOT_DIR = "/home/clarafercas/dynawo-notebooks/OpenModelica_only_users"
ROOT_DIR = isdir(joinpath(DEFAULT_ROOT_DIR, "Initialization")) ? DEFAULT_ROOT_DIR : pwd()

# Generated initialized package to validate.
# This defaults to the working copy produced by the initialization notebook.
# To compare another generated package instead, point this to its directory.
DEFAULT_AUX_PACKAGE_DIR = "/home/clarafercas/dynawo-notebooks/OpenModelica_only_users/Initialization/MyIEEE14_initialized"
BUILD_AUX_PACKAGE_DIR = joinpath(ROOT_DIR, "Initialization", "MyIEEE14_initialized")
AUX_PACKAGE_DIR = isdir(DEFAULT_AUX_PACKAGE_DIR) ? DEFAULT_AUX_PACKAGE_DIR : BUILD_AUX_PACKAGE_DIR
AUX_PACKAGE_FILE = joinpath(AUX_PACKAGE_DIR, "package.mo")
AUX_MODEL = "MyIEEE14_initialized.IEEE14DisconnectLine_initialized"

# Dynawo reference model
REFERENCE_PACKAGE_FILE = "/home/clarafercas/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/package.mo"
REFERENCE_MODEL = "Dynawo.Examples.IEEE14.TestCases.IEEE14DisconnectLine"

# Libraries
MODELICA_PKG_PATH = "/home/clarafercas/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo"
DYNAWO_PKG_PATH = REFERENCE_PACKAGE_FILE

# Comparison settings
STOP_TIME = 2000.0
MY_SLACK = "Gen1"
REFERENCE_SLACK = "Gen1"

println("Initialized package: ", AUX_PACKAGE_FILE)
println("Initialized model:   ", AUX_MODEL)
println("Reference model:     ", REFERENCE_MODEL)


In [ ]:
function om_send(omc, expr; parsed = true)
    println("OMC -> ", expr)
    try
        return sendExpression(omc, expr; parsed = parsed)
    catch err
        println(sendExpression(omc, "getErrorString()", parsed = false))
        rethrow(err)
    end
end

function get_result_variable_names(omc, resultfile::String)
    vars = sendExpression(omc, "readSimulationResultVars(\"$resultfile\")")
    return sort!(String.(vars))
end

function collect_component_names_from_results(omc, resultfile::String, suffix::String; prefix = nothing, pattern = nothing)
    names = String[]
    seen = Set{String}()

    for var in get_result_variable_names(omc, resultfile)
        endswith(var, suffix) || continue
        name = chopsuffix(var, suffix)
        !isnothing(prefix) && !startswith(name, prefix) && continue
        !isnothing(pattern) && !occursin(pattern, name) && continue
        name in seen && continue
        push!(seen, name)
        push!(names, name)
    end

    sort!(names)
    return names
end

function read_last_value(sys, full_name::String)
    values = getSolutions(sys, full_name)
    series = values[1]
    isempty(series) && error("No values found for $full_name")
    return Float64(series[end])
end

function read_complex(sys, base_name::String)
    re = read_last_value(sys, base_name * ".re")
    im = read_last_value(sys, base_name * ".im")
    return complex(re, im)
end

function voltage_mag_angle_deg_from_connector(sys, connector_path::String)
    v = read_complex(sys, connector_path * ".V")
    return abs(v), rad2deg(atan(imag(v), real(v)))
end

function terminal_power_pu(sys, component::String; terminal::String = "terminal")
    v = read_complex(sys, "$component.$terminal.V")
    i = read_complex(sys, "$component.$terminal.i")
    s = v * conj(i)
    return real(s), imag(s)
end

function run_and_simulate(label::String, package_file::String, model_name::String; stop_time::Float64 = STOP_TIME)
    isfile(package_file) || error("Package file not found: $package_file")

    omc = OMJulia.OMCSession()
    om_send(omc, "loadFile(\"$MODELICA_PKG_PATH\")")
    om_send(omc, "loadModel(Complex)")
    om_send(omc, "loadModel(ModelicaServices)")
    om_send(omc, "loadFile(\"$DYNAWO_PKG_PATH\")")
    package_file == DYNAWO_PKG_PATH || om_send(omc, "loadFile(\"$package_file\")")

    println("\nChecking $label...")
    chk = om_send(omc, "checkModel($model_name)", parsed = false)
    println(chk)

    build_dir = mktempdir()
    resultfile_name = replace(lowercase(label), " " => "_") * "_res.mat"

    ModelicaSystem(
        omc,
        package_file,
        model_name,
        [MODELICA_PKG_PATH, DYNAWO_PKG_PATH],
        customBuildDirectory = build_dir,
    )

    setSimulationOptions(omc, ["startTime=0", "stopTime=$(stop_time)", "stepSize=0.005", "tolerance=1e-6"])
    simulate(omc, resultfile = resultfile_name, simflags = "-s=euler -ls=klu -nls=kinsol")
    resultfile_path = joinpath(getWorkDirectory(omc), resultfile_name)

    return Dict(
        "label" => label,
        "omc" => omc,
        "resultfile" => resultfile_path,
        "build_dir" => build_dir,
        "model" => model_name,
    )
end


In [ ]:
function build_count_table(aux_counts::Dict{String,Int}, ref_counts::Dict{String,Int})
    component_types = ["buses", "non-slack generators", "transformers", "loads", "lines"]
    return DataFrame(
        component_type = component_types,
        auxiliary_count = [aux_counts[k] for k in component_types],
        dynawo_count = [ref_counts[k] for k in component_types],
        difference = [aux_counts[k] - ref_counts[k] for k in component_types],
    )
end

function build_voltage_table(aux_sys, ref_sys, components::Vector{String}; terminal::String = "terminal")
    rows = NamedTuple[]
    for component in sort(components)
        aux_u, aux_angle = voltage_mag_angle_deg_from_connector(aux_sys, "$component.$terminal")
        ref_u, ref_angle = voltage_mag_angle_deg_from_connector(ref_sys, "$component.$terminal")
        push!(rows, (
            component = component,
            auxiliary_u_pu = aux_u,
            dynawo_u_pu = ref_u,
            delta_u_pu = aux_u - ref_u,
            abs_delta_u_pu = abs(aux_u - ref_u),
            auxiliary_angle_deg = aux_angle,
            dynawo_angle_deg = ref_angle,
            delta_angle_deg = aux_angle - ref_angle,
            abs_delta_angle_deg = abs(aux_angle - ref_angle),
        ))
    end
    return DataFrame(rows)
end

function build_terminal_power_table(aux_sys, ref_sys, components::Vector{String}; terminal::String = "terminal")
    rows = NamedTuple[]
    for component in sort(components)
        aux_p, aux_q = terminal_power_pu(aux_sys, component; terminal = terminal)
        ref_p, ref_q = terminal_power_pu(ref_sys, component; terminal = terminal)
        push!(rows, (
            component = component,
            auxiliary_p_pu = aux_p,
            dynawo_p_pu = ref_p,
            delta_p_pu = aux_p - ref_p,
            abs_delta_p_pu = abs(aux_p - ref_p),
            auxiliary_q_pu = aux_q,
            dynawo_q_pu = ref_q,
            delta_q_pu = aux_q - ref_q,
            abs_delta_q_pu = abs(aux_q - ref_q),
        ))
    end
    return DataFrame(rows)
end

function max_abs_or_zero(df::DataFrame, col::Symbol)
    nrow(df) == 0 && return 0.0
    return maximum(df[!, col])
end

function build_summary_table(bus_voltage_df::DataFrame, generator_power_df::DataFrame, transformer_power_df::DataFrame, load_power_df::DataFrame, line_power_df::DataFrame, slack_power_df::DataFrame)
    return DataFrame(
        quantity = [
            "max |delta bus U| pu",
            "max |delta bus angle| deg",
            "max |delta generator P| pu",
            "max |delta generator Q| pu",
            "max |delta transformer terminal1 P| pu",
            "max |delta transformer terminal1 Q| pu",
            "max |delta load P| pu",
            "max |delta load Q| pu",
            "max |delta line terminal1 P| pu",
            "max |delta line terminal1 Q| pu",
            "|delta slack P| pu",
            "|delta slack Q| pu",
        ],
        value = [
            max_abs_or_zero(bus_voltage_df, :abs_delta_u_pu),
            max_abs_or_zero(bus_voltage_df, :abs_delta_angle_deg),
            max_abs_or_zero(generator_power_df, :abs_delta_p_pu),
            max_abs_or_zero(generator_power_df, :abs_delta_q_pu),
            max_abs_or_zero(transformer_power_df, :abs_delta_p_pu),
            max_abs_or_zero(transformer_power_df, :abs_delta_q_pu),
            max_abs_or_zero(load_power_df, :abs_delta_p_pu),
            max_abs_or_zero(load_power_df, :abs_delta_q_pu),
            max_abs_or_zero(line_power_df, :abs_delta_p_pu),
            max_abs_or_zero(line_power_df, :abs_delta_q_pu),
            max_abs_or_zero(slack_power_df, :abs_delta_p_pu),
            max_abs_or_zero(slack_power_df, :abs_delta_q_pu),
        ],
    )
end

function add_power_discrepancies!(rows::Vector{NamedTuple}, component_type::String, df::DataFrame)
    for row in eachrow(df)
        push!(rows, (
            component_type = component_type,
            component = row.component,
            score = max(row.abs_delta_p_pu, row.abs_delta_q_pu),
            delta_p_pu = row.delta_p_pu,
            delta_q_pu = row.delta_q_pu,
            delta_u_pu = missing,
            delta_angle_deg = missing,
        ))
    end
end

function add_voltage_discrepancies!(rows::Vector{NamedTuple}, component_type::String, df::DataFrame)
    for row in eachrow(df)
        push!(rows, (
            component_type = component_type,
            component = row.component,
            score = max(row.abs_delta_u_pu, row.abs_delta_angle_deg / 100),
            delta_p_pu = missing,
            delta_q_pu = missing,
            delta_u_pu = row.delta_u_pu,
            delta_angle_deg = row.delta_angle_deg,
        ))
    end
end

function build_top_discrepancy_table(bus_voltage_df::DataFrame, generator_power_df::DataFrame, transformer_power_df::DataFrame, load_power_df::DataFrame, line_power_df::DataFrame, slack_power_df::DataFrame; n::Int = 12)
    rows = NamedTuple[]
    add_voltage_discrepancies!(rows, "bus voltage", bus_voltage_df)
    add_power_discrepancies!(rows, "generator terminal power", generator_power_df)
    add_power_discrepancies!(rows, "transformer terminal1 power", transformer_power_df)
    add_power_discrepancies!(rows, "load terminal power", load_power_df)
    add_power_discrepancies!(rows, "line terminal1 power", line_power_df)
    add_power_discrepancies!(rows, "slack terminal power", slack_power_df)

    isempty(rows) && return DataFrame()
    df = DataFrame(rows)
    sort!(df, :score, rev = true)
    return first(df, min(n, nrow(df)))
end


In [ ]:
aux_run = run_and_simulate("MyIEEE14 initialized", AUX_PACKAGE_FILE, AUX_MODEL; stop_time = STOP_TIME)
ref_run = run_and_simulate("Dynawo IEEE14DisconnectLine", REFERENCE_PACKAGE_FILE, REFERENCE_MODEL; stop_time = STOP_TIME)

aux_omc = aux_run["omc"]
ref_omc = ref_run["omc"]

println("\nInitialized result file: ", aux_run["resultfile"])
println("Reference result file: ", ref_run["resultfile"])


In [ ]:
aux_buses = collect_component_names_from_results(aux_omc, aux_run["resultfile"], ".terminal.V.re"; prefix = "Bus")
ref_buses = collect_component_names_from_results(ref_omc, ref_run["resultfile"], ".terminal.V.re"; prefix = "Bus")
common_buses = sort(intersect(aux_buses, ref_buses))

aux_generators = setdiff(collect_component_names_from_results(aux_omc, aux_run["resultfile"], ".terminal.V.re"; prefix = "Gen"), [MY_SLACK])
ref_generators = setdiff(collect_component_names_from_results(ref_omc, ref_run["resultfile"], ".terminal.V.re"; prefix = "Gen"), [REFERENCE_SLACK])
common_generators = sort(intersect(aux_generators, ref_generators))

aux_transformers = collect_component_names_from_results(aux_omc, aux_run["resultfile"], ".terminal1.V.re"; prefix = "Tfo")
ref_transformers = collect_component_names_from_results(ref_omc, ref_run["resultfile"], ".terminal1.V.re"; prefix = "Tfo")
common_transformers = sort(intersect(aux_transformers, ref_transformers))

aux_loads = collect_component_names_from_results(aux_omc, aux_run["resultfile"], ".terminal.V.re"; prefix = "Load")
ref_loads = collect_component_names_from_results(ref_omc, ref_run["resultfile"], ".terminal.V.re"; prefix = "Load")
common_loads = sort(intersect(aux_loads, ref_loads))

aux_lines = collect_component_names_from_results(aux_omc, aux_run["resultfile"], ".terminal1.V.re"; prefix = "Line")
ref_lines = collect_component_names_from_results(ref_omc, ref_run["resultfile"], ".terminal1.V.re"; prefix = "Line")
common_lines = sort(intersect(aux_lines, ref_lines))

count_comparison_df = build_count_table(
    Dict(
        "buses" => length(aux_buses),
        "non-slack generators" => length(aux_generators),
        "transformers" => length(aux_transformers),
        "loads" => length(aux_loads),
        "lines" => length(aux_lines),
    ),
    Dict(
        "buses" => length(ref_buses),
        "non-slack generators" => length(ref_generators),
        "transformers" => length(ref_transformers),
        "loads" => length(ref_loads),
        "lines" => length(ref_lines),
    ),
)

bus_voltage_df = build_voltage_table(aux_omc, ref_omc, common_buses)
generator_power_df = build_terminal_power_table(aux_omc, ref_omc, common_generators)
transformer_power_df = build_terminal_power_table(aux_omc, ref_omc, common_transformers; terminal = "terminal1")
load_power_df = build_terminal_power_table(aux_omc, ref_omc, common_loads)
line_power_df = build_terminal_power_table(aux_omc, ref_omc, common_lines; terminal = "terminal1")
slack_power_df = build_terminal_power_table(aux_omc, ref_omc, [MY_SLACK])

summary_df = build_summary_table(bus_voltage_df, generator_power_df, transformer_power_df, load_power_df, line_power_df, slack_power_df)
top_discrepancy_df = build_top_discrepancy_table(bus_voltage_df, generator_power_df, transformer_power_df, load_power_df, line_power_df, slack_power_df)


In [ ]:
println("Counts in both models:")
display(count_comparison_df)

println("Max absolute differences:")
display(summary_df)

println("Top discrepancies across buses, generators, transformers, loads, lines, and slack:")
display(top_discrepancy_df)

println("Non-slack generator terminal P/Q:")
display(generator_power_df[:, [:component, :auxiliary_p_pu, :dynawo_p_pu, :delta_p_pu, :auxiliary_q_pu, :dynawo_q_pu, :delta_q_pu]])

println("Slack terminal P/Q:")
display(slack_power_df[:, [:component, :auxiliary_p_pu, :dynawo_p_pu, :delta_p_pu, :auxiliary_q_pu, :dynawo_q_pu, :delta_q_pu]])

println("Bus voltage magnitude and angle:")
display(bus_voltage_df[:, [:component, :auxiliary_u_pu, :dynawo_u_pu, :delta_u_pu, :auxiliary_angle_deg, :dynawo_angle_deg, :delta_angle_deg]])

println("Load terminal P/Q:")
display(load_power_df[:, [:component, :auxiliary_p_pu, :dynawo_p_pu, :delta_p_pu, :auxiliary_q_pu, :dynawo_q_pu, :delta_q_pu]])


## Notes

This comparison uses the initialized user-style model against the Dynawo library example with the same disconnect-line event. Small numerical differences can still appear because the two simulations may not emit exactly the same output grid or may follow slightly different internal initialization paths, even when the electrical trajectories match closely.
